# Exiting muon toy study: MCS vs MCS+CSDA

Goal: evaluate whether combining CSDA information with MCS helps **partially contained (exiting) muons**.

We compare energy resolution for:
- **MCS-only fit**
- **MCS+CSDA combined fit**


## Mathematical model

For observed scattering angles $\{\theta_i\}_{i=1}^N$ along the contained track segment, MCS-only fit minimizes:

$$
\hat T_{\mathrm{MCS}} = \arg\min_{T_0}\;\mathcal{L}_{\mathrm{MCS}}(T_0)
$$

where $\mathcal{L}_{\mathrm{MCS}}$ is the negative log-likelihood from the Highland+detector-resolution model in `spine.utils.mcs`.

For an exiting track, the measured contained length $L_c$ does not give full range, but it still provides a CSDA-based proxy energy:

$$
T_{\mathrm{CSDA,proxy}} = f_{\mathrm{CSDA}}(L_c)
$$

We combine both via:

$$
\hat T_{\mathrm{comb}} = \arg\min_{T_0}\left[
\mathcal{L}_{\mathrm{MCS}}(T_0)
+ \frac{w}{2}\left(\frac{T_0-T_{\mathrm{CSDA,proxy}}}{\sigma_{\mathrm{CSDA}}}\right)^2
\right]
$$

with

$$
\sigma_{\mathrm{CSDA}} = \max\left(1,\;f\,T_{\mathrm{CSDA,proxy}}\right)
$$

This regularizes catastrophic MCS-only failures while still allowing MCS information to move the estimate.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from spine.utils.energy_loss import csda_range_lar, csda_table_spline, step_energy_loss_lar
from spine.utils.globals import MUON_MASS, MUON_PID
from spine.utils.mcs import highland, mcs_fit

rng = np.random.default_rng(20260409)
csda_spline = csda_table_spline(MUON_PID)

plt.style.use('seaborn-v0_8-whitegrid')


## Toy setup (exiting muons)

- True initial kinetic energies: **200, 400, 600, 800, 1000 MeV**
- Contained fraction of full CSDA range: **0.2, 0.4, 0.6, 0.8**
- Segment size: **5 cm**
- 300 toys per (energy, fraction)
- Contained-length measurement smear: 3 cm


In [ ]:
def simulate_exiting_muon(T0_true, frac_contained, dx=5.0, length_smear_cm=3.0):
    # True full range and observed (contained) length before exiting detector
    R_true = csda_range_lar(T0_true, MUON_MASS)
    L_cont_true = frac_contained * R_true
    L_cont_reco = max(0.0, L_cont_true + rng.normal(0.0, length_smear_cm))

    # Build MCS angles from the contained segment only
    n_steps = int(np.floor(L_cont_true / dx))
    if n_steps < 4:
        return None

    ke = step_energy_loss_lar(T0_true, MUON_MASS, dx, num_steps=n_steps)
    if len(ke) < n_steps + 1:
        return None

    p = np.sqrt(ke**2 + 2.0 * MUON_MASS * ke)
    p_steps = np.sqrt(p[:-1] * p[1:])
    theta0 = highland(p_steps, MUON_MASS, dx)

    # Match fitter default detector angular resolution model
    res = 0.25 / dx**1.25
    theta_obs = rng.rayleigh(np.sqrt(theta0**2 + res**2))

    # CSDA proxy from contained length
    csda_ke_proxy = float(csda_spline(L_cont_reco))

    # Fits
    ke_mcs = mcs_fit(theta_obs, MUON_MASS, dx, upper_bound=1200.0)
    ke_comb = mcs_fit(
        theta_obs,
        MUON_MASS,
        dx,
        upper_bound=1200.0,
        csda_ke=csda_ke_proxy,
        csda_ke_frac=0.30,
        csda_weight=2.0,
        csda_as_lower_bound=False,
    )

    return {
        'T0_true': T0_true,
        'frac_contained': frac_contained,
        'R_true': R_true,
        'L_cont_true': L_cont_true,
        'L_cont_reco': L_cont_reco,
        'n_steps': n_steps,
        'csda_ke_proxy': csda_ke_proxy,
        'mcs_ke': ke_mcs,
        'comb_ke': ke_comb,
    }


def robust_resolution(x):
    q75, q25 = np.percentile(x, [75, 25])
    return (q75 - q25) / 1.349  # Gaussian-equivalent sigma estimate


In [ ]:
energies = np.array([200.0, 400.0, 600.0, 800.0, 1000.0])
fractions = np.array([0.2, 0.4, 0.6, 0.8])
n_toys = 300

records = []
for T0 in energies:
    for frac in fractions:
        for _ in range(n_toys):
            rec = simulate_exiting_muon(T0, frac)
            if rec is not None:
                records.append(rec)

df = pd.DataFrame(records)
df['mcs_err'] = df['mcs_ke'] - df['T0_true']
df['comb_err'] = df['comb_ke'] - df['T0_true']

print(f"kept toys: {len(df)}")
df.head()


In [ ]:
summary_rows = []
for T0 in energies:
    for frac in fractions:
        d = df[(df.T0_true == T0) & (df.frac_contained == frac)]
        summary_rows.append({
            'T0_true': T0,
            'frac_contained': frac,
            'N': len(d),
            'mcs_bias': d['mcs_err'].mean(),
            'comb_bias': d['comb_err'].mean(),
            'mcs_res': robust_resolution(d['mcs_err'].values),
            'comb_res': robust_resolution(d['comb_err'].values),
        })

summary = pd.DataFrame(summary_rows)
summary['res_improvement_pct'] = 100.0 * (summary['mcs_res'] - summary['comb_res']) / summary['mcs_res']
summary


In [ ]:
# Global metrics over the full scan
for label, col in [('MCS only','mcs_err'), ('MCS+CSDA','comb_err')]:
    err = df[col].values
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    print(f"{label:10s}  MAE={mae:8.2f} MeV   RMSE={rmse:8.2f} MeV")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for T0 in energies:
    d = summary[summary.T0_true == T0]
    axes[0].plot(d['frac_contained'], d['mcs_res'], marker='o', label=f'MCS {int(T0)} MeV')
    axes[0].plot(d['frac_contained'], d['comb_res'], marker='s', linestyle='--', label=f'Comb {int(T0)} MeV')

axes[0].set_title('Resolution vs contained fraction')
axes[0].set_xlabel('Contained fraction of full range')
axes[0].set_ylabel('Robust sigma of (fit - true) [MeV]')
axes[0].legend(ncols=2, fontsize=8)

for T0 in energies:
    d = summary[summary.T0_true == T0]
    axes[1].plot(d['frac_contained'], d['res_improvement_pct'], marker='o', label=f'{int(T0)} MeV')

axes[1].axhline(0.0, color='k', lw=1)
axes[1].set_title('Relative resolution gain of MCS+CSDA')
axes[1].set_xlabel('Contained fraction of full range')
axes[1].set_ylabel('Resolution improvement [%]')
axes[1].legend(title='True KE', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Residual distributions at a representative contained fraction
frac_show = 0.4
fig, axes = plt.subplots(len(energies), 1, figsize=(9, 2.1*len(energies)), sharex=True)

for i, T0 in enumerate(energies):
    d = df[(df.T0_true == T0) & (df.frac_contained == frac_show)]
    axes[i].hist(d['mcs_err'], bins=40, alpha=0.6, label='MCS')
    axes[i].hist(d['comb_err'], bins=40, alpha=0.6, label='MCS+CSDA')
    axes[i].axvline(0.0, color='k', lw=1)
    axes[i].set_ylabel(f'{int(T0)} MeV')
    if i == 0:
        axes[i].legend(fontsize=8)

axes[-1].set_xlabel(f'Residual (fit - true) [MeV], contained fraction={frac_show}')
plt.tight_layout()
plt.show()


## Notes

- This notebook intentionally studies **exiting muons**, not contained stopping muons.
- We scan multiple true energies and multiple contained lengths, as requested.
- The key check is whether the hybrid estimator improves resolution compared to MCS-only across this grid.


## Precomputed results (executed and saved)

The toy simulation was re-run after auditing `mcs_fit` settings. To avoid pathological high-tail MCS fits in this toy setup, the fit upper bound is fixed to **1200 MeV** (scan max true KE is 1000 MeV).

Global metrics from this saved run:

- MCS-only: MAE = 215.78 MeV, RMSE = 314.20 MeV
- MCS+CSDA: MAE = 211.65 MeV, RMSE = 285.11 MeV

Average robust resolution by contained fraction:

| Contained fraction | MCS robust sigma [MeV] | MCS+CSDA robust sigma [MeV] | Improvement [%] |
|---:|---:|---:|---:|
| 0.2 | 552.8 | 8.3 | 98.4 |
| 0.4 | 367.6 | 13.0 | 96.2 |
| 0.6 | 227.1 | 22.4 | 88.9 |
| 0.8 | 103.8 | 34.2 | 63.7 |

Saved plots:

![Resolution vs fraction](artifacts/resolution_vs_fraction.svg)

![Resolution improvement heatmap](artifacts/resolution_improvement_heatmap.svg)
